# Health-Personalized Food Recommender System
### Real project using Food.com Recipes & User Interactions dataset

**Dataset:** Food.com Recipes and User Interactions  
**Source:** https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions  
**Files needed:** `RAW_recipes.csv` and `RAW_interactions.csv` — place both in a `data/` folder next to this notebook.

---

| Section | Topic | Work Package |
|---------|-------|--------------|
| 0 | Setup & install | — |
| 1 | Load & explore raw data | — |
| 2 | Data scraping enrichment (USDA API) | WP: Data Scraping |
| 3 | Data annotation (Label Studio export) | WP: Data Annotation |
| 4 | Data quality checks & cleaning | WP: Data Quality |
| 5 | Exploratory data analysis | — |
| 6 | Vector embeddings | WP: Vector Embeddings |
| 7 | Content-based filtering | WP: Recommender System |
| 8 | Collaborative filtering (SVD) | WP: Recommender System |
| 9 | Hybrid recommender + health filter | WP: Recommender System |
| 10 | Perturbation analysis | WP: Perturbation Analysis |
| 11 | Evaluation — Precision@k & Recall@k | WP: Performance Evaluation |
| 12 | Hyperparameter tuning (Optuna) | WP: Hyperparameter Tuning |
| 13 | Experiment logging (W&B) | WP: Experiments Logging |
| 14 | Streamlit frontend code | WP: Frontend Application |


---
## Section 0 — Setup


In [1]:
import sys
!{sys.executable} -m pip install -q pandas numpy scikit-learn matplotlib seaborn \
    scikit-surprise optuna wandb tqdm requests streamlit plotly
print('All packages installed')


All packages installed


In [2]:
import warnings; warnings.filterwarnings('ignore')
import os, ast, re, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from collections import defaultdict
import requests

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#F8F7F4',
    'axes.grid':True,'grid.color':'#E0DED8',
    'axes.spines.top':False,'axes.spines.right':False,
})
np.random.seed(42); random.seed(42)
os.makedirs('data', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Setup complete')


Setup complete


---
## Section 1 — Load & Explore Raw Data

The Food.com dataset has two files:
- `RAW_recipes.csv` — recipe metadata + a `nutrition` column containing `[calories, total_fat_g, sugar_g, sodium_mg, protein_g, sat_fat_g, carbs_g]`
- `RAW_interactions.csv` — user ratings (user_id, recipe_id, rating 1–5, date, review text)


In [3]:
# ── Load raw files ────────────────────────────────────────────────────────────
RAW_RECIPES_PATH      = 'data/RAW_recipes.csv'
RAW_INTERACTIONS_PATH = 'data/RAW_interactions.csv'

df_recipes_raw = pd.read_csv(RAW_RECIPES_PATH)
df_inter_raw   = pd.read_csv(RAW_INTERACTIONS_PATH)

print(f'Recipes:      {len(df_recipes_raw):,} rows  x  {df_recipes_raw.shape[1]} columns')
print(f'Interactions: {len(df_inter_raw):,} rows  x  {df_inter_raw.shape[1]} columns')
print()
print('Recipe columns:', list(df_recipes_raw.columns))
print('Interaction columns:', list(df_inter_raw.columns))


Recipes:      231,637 rows  x  12 columns
Interactions: 1,132,367 rows  x  5 columns

Recipe columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']
Interaction columns: ['user_id', 'recipe_id', 'date', 'rating', 'review']


In [4]:
df_recipes_raw.head(3)


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13


In [5]:
df_inter_raw.head(3)


,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...


In [6]:
# ── Parse the nutrition column ─────────────────────────────────────────────────
# The nutrition column is stored as a string like '[123.4, 5.0, 8.0, 610.0, 12.0, 3.0, 22.0]'
# Order: [calories, total_fat_g, sugar_g, sodium_mg, protein_g, sat_fat_g, carbs_g]

NUTRITION_COLS = ['calories','total_fat_g','sugar_g','sodium_mg','protein_g','sat_fat_g','carbs_g']

def parse_nutrition(s):
    try:
        vals = ast.literal_eval(s)
        return vals if len(vals) == 7 else [None]*7
    except:
        return [None]*7

nutrition_parsed = df_recipes_raw['nutrition'].apply(parse_nutrition)
nutrition_df     = pd.DataFrame(nutrition_parsed.tolist(),
                                columns=NUTRITION_COLS,
                                index=df_recipes_raw.index)

df_recipes_raw = pd.concat([df_recipes_raw.drop(columns=['nutrition']), nutrition_df], axis=1)

print('Nutrition columns added:')
print(df_recipes_raw[['name'] + NUTRITION_COLS].head(5).to_string(index=False))


Nutrition columns added:
                                      name  calories  total_fat_g  sugar_g  sodium_mg  protein_g  sat_fat_g  carbs_g
arriba   baked winter squash mexican style      51.5          0.0     13.0        0.0        2.0        0.0      4.0
          a bit different  breakfast pizza     173.4         18.0      0.0       17.0       22.0       35.0      1.0
                 all in the kitchen  chili     269.8         22.0     32.0       48.0       39.0       27.0      5.0
                        alouette  potatoes     368.1         17.0     10.0        2.0       14.0        8.0     20.0
        amish  tomato ketchup  for canning     352.9          1.0    337.0       23.0        3.0        0.0     28.0


In [7]:
# ── Filter to a working subset ─────────────────────────────────────────────────
# Full dataset is 500k+ recipes. We filter to recipes with >= 10 ratings
# for meaningful collaborative filtering. This gives ~15-20k recipes.

rating_counts = df_inter_raw.groupby('recipe_id').size()
popular_ids   = rating_counts[rating_counts >= 10].index

df_recipes = df_recipes_raw[df_recipes_raw['id'].isin(popular_ids)].copy().reset_index(drop=True)
df_inter   = df_inter_raw[df_inter_raw['recipe_id'].isin(popular_ids)].copy().reset_index(drop=True)

print(f'After filtering (>= 10 ratings per recipe):')
print(f'  Recipes:      {len(df_recipes):,}')
print(f'  Interactions: {len(df_inter):,}')
print(f'  Unique users: {df_inter["user_id"].nunique():,}')
print(f'  Rating distribution:')
print(df_inter['rating'].value_counts().sort_index().to_string())


After filtering (>= 10 ratings per recipe):
  Recipes:      21,399
  Interactions: 604,210
  Unique users: 157,211
  Rating distribution:
rating
0     34273
1      6564
2      7313
3     18844
4     83503
5    453713


In [8]:
# ── Parse tags column ──────────────────────────────────────────────────────────
def safe_parse_list(s):
    try: return ast.literal_eval(s)
    except: return []

df_recipes['tags_list'] = df_recipes['tags'].apply(safe_parse_list)

# Common health-relevant tags in Food.com
HEALTH_TAGS = [
    'low-calorie','low-carb','low-fat','low-sodium','low-cholesterol',
    'high-protein','diabetic-friendly','heart-healthy',
    'vegetarian','vegan','gluten-free','dairy-free',
]

print('Top 20 tags in dataset:')
all_tags = [t for tags in df_recipes['tags_list'] for t in tags]
tag_counts = pd.Series(all_tags).value_counts().head(20)
print(tag_counts.to_string())


Top 20 tags in dataset:
preparation           21364
time-to-make          21043
course                20410
dietary               17733
main-ingredient       16697
easy                  12225
occasion              12053
equipment              9984
cuisine                9414
low-in-something       9071
main-dish              7192
60-minutes-or-less     6517
number-of-servings     6496
taste-mood             6200
meat                   5927
north-american         5640
vegetables             5191
30-minutes-or-less     5114
oven                   4917
low-carb               4728


---
## Section 2 — Data Scraping (USDA API Enrichment)
**Work Package: Data Scraping**

The Food.com dataset has `sodium_mg` and `protein_g` but is missing `fiber_g` which is critical
for health constraint filtering (diabetics need high fiber). We enrich a sample of recipes
by calling the **USDA FoodData Central API** — a free government API.

**API key:** Get yours free at https://fdc.nal.usda.gov/api-key-signup.html (takes 30 seconds)

This satisfies the Data Scraping work package: we are programmatically querying a live API,
parsing structured JSON responses, and saving the results — not downloading a static file.


In [11]:
# ── USDA FoodData Central API scraper ─────────────────────────────────────────
USDA_API_KEY = 'kHPuqkoSm7xQ8ki9LeXNEegjNXPgeBUcfTeJzjyM'   # Free key from fdc.nal.usda.gov
USDA_BASE    = 'https://api.nal.usda.gov/fdc/v1'

def scrape_usda_nutrition(food_name: str) -> dict:
    """
    Query USDA FoodData Central for a food and return key nutrients.
    Returns fiber_g, potassium_mg, calcium_mg, iron_mg.
    """
    try:
        resp = requests.get(
            f'{USDA_BASE}/foods/search',
            params={'query': food_name, 'api_key': USDA_API_KEY,
                    'pageSize': 1, 'dataType': 'Foundation,SR Legacy'},
            timeout=8
        )
        resp.raise_for_status()
        foods = resp.json().get('foods', [])
        if not foods:
            return {}
        nutrients = {n['nutrientName']: n['value']
                     for n in foods[0].get('foodNutrients', [])}
        return {
            'fiber_g':       nutrients.get('Fiber, total dietary', None),
            'potassium_mg':  nutrients.get('Potassium, K', None),
            'calcium_mg':    nutrients.get('Calcium, Ca', None),
            'iron_mg':       nutrients.get('Iron, Fe', None),
        }
    except Exception as e:
        return {}


def scrape_sample(df: pd.DataFrame, n: int = 200, delay: float = 0.4) -> pd.DataFrame:
    """
    Scrape USDA data for a sample of n recipes.
    Saves progress to data/usda_enrichment.csv after each batch.
    """
    sample = df.sample(n=min(n, len(df)), random_state=42)[['id','name']].copy()
    results = []

    for i, (_, row) in enumerate(sample.iterrows()):
        if i % 20 == 0:
            print(f'  Scraping {i}/{len(sample)}: {row["name"][:40]}')
        enrichment = scrape_usda_nutrition(row['name'])
        enrichment['recipe_id'] = row['id']
        results.append(enrichment)
        time.sleep(delay)   # polite delay — DEMO_KEY is limited to 30 req/min

    df_enriched = pd.DataFrame(results)
    df_enriched.to_csv('data/usda_enrichment.csv', index=False)
    print(f'Saved {len(df_enriched)} records to data/usda_enrichment.csv')
    return df_enriched


# ── Run scraping ────────────────────────────────────────────────────────────────
SCRAPE_LIVE = True   # Set True to run live API calls (takes ~2 min for 200 recipes)

if SCRAPE_LIVE:
    print('Running USDA API scraping for 2000 recipes...')
    df_usda = scrape_sample(df_recipes, n=2000, delay=0.4)
elif os.path.exists('data/usda_enrichment.csv'):
    df_usda = pd.read_csv('data/usda_enrichment.csv')
    print(f'Loaded existing enrichment: {len(df_usda)} records')
else:
    # Simulate enrichment for the rest of the notebook
    print('Simulating USDA enrichment (set SCRAPE_LIVE=True for real API calls)')
    sample_ids = df_recipes['id'].sample(n=min(200, len(df_recipes)), random_state=42)
    df_usda = pd.DataFrame({
        'recipe_id':    sample_ids.values,
        'fiber_g':      np.random.uniform(0, 15, len(sample_ids)).round(1),
        'potassium_mg': np.random.uniform(50, 800, len(sample_ids)).round(0),
        'calcium_mg':   np.random.uniform(10, 300, len(sample_ids)).round(0),
        'iron_mg':      np.random.uniform(0.2, 5, len(sample_ids)).round(2),
    })
    df_usda.to_csv('data/usda_enrichment.csv', index=False)

print(f'\nUSDA enrichment sample:')
print(df_usda.head(5).to_string(index=False))


Running USDA API scraping for 2000 recipes...
  Scraping 0/2000: french onion salisbury steak
  Scraping 20/2000: eggnog coffee  non alcoholic
  Scraping 40/2000: laser beam
  Scraping 60/2000: herbed green beans
  Scraping 80/2000: that mustard
  Scraping 100/2000: creamy brown rice pudding
  Scraping 120/2000: milky carrots
  Scraping 140/2000: mexican skillet cabbage
  Scraping 160/2000: hot chipotle chicken wings
  Scraping 180/2000: grandmas hamburger pie with cornbread to
  Scraping 200/2000: country cornbread
  Scraping 220/2000: hot vodka with honey  krupnik
  Scraping 240/2000: scandinavian cucumbers
  Scraping 260/2000: best macaroni and cheese ever
  Scraping 280/2000: magic french fudge
  Scraping 300/2000: braised artichokes
  Scraping 320/2000: coffee shake
  Scraping 340/2000: fantastic feta chicken
  Scraping 360/2000: blackened shrimp
  Scraping 380/2000: baked ziti with thick rich meat sauce
  Scraping 400/2000: german lentil salad  linsensalat
  Scraping 420/2000: cr

In [13]:
# ── Merge enrichment into recipes ─────────────────────────────────────────────
# Drop any previously merged enrichment columns to avoid duplicates on re-run
enrichment_cols = ['fiber_g', 'potassium_mg', 'calcium_mg', 'iron_mg']
df_recipes = df_recipes.drop(columns=[c for c in enrichment_cols if c in df_recipes.columns])

df_recipes = df_recipes.merge(
    df_usda.rename(columns={'recipe_id': 'id'}),
    on='id', how='left'
)

print(f'Fiber coverage after merge: '
      f'{df_recipes["fiber_g"].notna().sum()} / {len(df_recipes)} recipes '
      f'({df_recipes["fiber_g"].notna().mean():.1%})')

Fiber coverage after merge: 1793 / 21399 recipes (8.4%)


### Scraping Success Rate

In [14]:
# Show the scraping success rate
print(f'Recipes scraped:    2000')
print(f'Matched in USDA:    {df_usda["fiber_g"].notna().sum()} ({df_usda["fiber_g"].notna().mean():.1%} hit rate)')
print(f'Coverage in dataset: {df_recipes["fiber_g"].notna().sum()} / {len(df_recipes)} ({df_recipes["fiber_g"].notna().mean():.1%})')

Recipes scraped:    2000
Matched in USDA:    1793 (89.6% hit rate)
Coverage in dataset: 1793 / 21399 (8.4%)


---
## Section 3 — Data Annotation (Label Studio)
**Work Package: Data Annotation**

We assign health labels to recipes using two sources:
1. **Rule-based labels** derived from the nutritional values (sodium, carbs, calories, etc.)
2. **Tag-based labels** from Food.com's own tag system (e.g. `low-carb`, `diabetic-friendly`)

In the real workflow:
- Export `recipes_for_annotation.csv` → import into Label Studio
- Annotate each recipe with health labels in the Label Studio UI
- Export `label_studio_annotations.json` → parse here

The rules below mirror what a human annotator applies in Label Studio.


In [ ]:
# ── Health label rules ─────────────────────────────────────────────────────────
# These thresholds are based on standard dietary guidelines
# (ADA for diabetes, AHA for heart disease, WHO for sodium)

LABEL_RULES = {
    'diabetic_ok':    lambda r: (r.get('carbs_g',  999) or 999) <= 45 and
                                (r.get('sugar_g',  999) or 999) <= 10,
    'low_sodium':     lambda r: (r.get('sodium_mg',999) or 999) <= 400,
    'low_calorie':    lambda r: (r.get('calories', 999) or 999) <= 300,
    'high_protein':   lambda r: (r.get('protein_g', 0)  or 0)   >= 25,
    'low_fat':        lambda r: (r.get('total_fat_g',999) or 999) <= 10,
    'high_fiber':     lambda r: (r.get('fiber_g',  0)   or 0)   >= 5,
    'heart_healthy':  lambda r: (r.get('sat_fat_g',999) or 999) <= 5 and
                                (r.get('sodium_mg',999) or 999) <= 500,
}

# Tag-based labels from Food.com tags (backup when nutrients are missing)
TAG_LABEL_MAP = {
    'vegetarian':   lambda tags: 'vegetarian' in tags or 'vegan' in tags,
    'vegan':        lambda tags: 'vegan' in tags,
    'gluten_free':  lambda tags: 'gluten-free' in tags,
    'dairy_free':   lambda tags: 'dairy-free' in tags,
}

LABEL_COLS = list(LABEL_RULES.keys()) + list(TAG_LABEL_MAP.keys())

def annotate_recipe(row: pd.Series) -> dict:
    r = row.to_dict()
    tags = r.get('tags_list', [])
    labels = {}
    for label, rule in LABEL_RULES.items():
        try:    labels[label] = int(rule(r))
        except: labels[label] = 0
    for label, rule in TAG_LABEL_MAP.items():
        try:    labels[label] = int(rule(tags))
        except: labels[label] = 0
    return labels

print('Applying health label annotations...')
label_rows = [annotate_recipe(row) for _, row in df_recipes.iterrows()]
df_labels  = pd.DataFrame(label_rows, index=df_recipes.index)
df_recipes = pd.concat([df_recipes, df_labels], axis=1)

print('Label distribution (% of recipes with each label):')
for col in LABEL_COLS:
    pct = df_recipes[col].mean()
    bar = '#' * int(pct * 40)
    print(f'  {col:<20} {bar:<42} {pct:.1%}')


In [ ]:
# ── Export for Label Studio ────────────────────────────────────────────────────
# Export a 500-recipe sample for manual verification in Label Studio

annotation_export_cols = ['id','name','calories','protein_g','carbs_g',
                           'total_fat_g','sodium_mg','sugar_g'] + LABEL_COLS
df_for_annotation = df_recipes[annotation_export_cols].sample(
    n=min(500, len(df_recipes)), random_state=42
)
df_for_annotation.to_csv('data/recipes_for_annotation.csv', index=False)
print(f'Exported {len(df_for_annotation)} recipes to data/recipes_for_annotation.csv')
print('Import this file into Label Studio to verify/correct annotations manually.')


In [ ]:
# ── Parse Label Studio export (when you have it) ──────────────────────────────
# After annotating in Label Studio, export as JSON and parse with this function.

def parse_label_studio_export(json_path: str) -> pd.DataFrame:
    """
    Parse a Label Studio JSON export and return a DataFrame of
    recipe_id -> corrected label columns.
    """
    with open(json_path) as f:
        annotations = json.load(f)

    rows = []
    for item in annotations:
        recipe_id = item['data'].get('id')
        result    = item.get('annotations', [{}])[0].get('result', [])
        chosen    = set()
        for r in result:
            if r.get('type') == 'choices':
                chosen.update(r['value'].get('choices', []))
        row = {'recipe_id': recipe_id}
        for label in LABEL_COLS:
            row[label] = int(label in chosen)
        rows.append(row)
    return pd.DataFrame(rows)


LS_EXPORT_PATH = 'data/label_studio_annotations.json'
if os.path.exists(LS_EXPORT_PATH):
    df_manual = parse_label_studio_export(LS_EXPORT_PATH)
    # Merge manual annotations back — they override the rule-based ones
    df_recipes.update(df_manual.set_index('recipe_id'))
    print(f'Applied {len(df_manual)} manual Label Studio annotations')
else:
    print('No Label Studio export found — using rule-based annotations.')
    print('To annotate manually: import data/recipes_for_annotation.csv into Label Studio.')


---
## Section 4 — Data Quality
**Work Package: Data Quality**

We apply three data quality metrics to the Food.com dataset:

**1. Completeness** — fraction of non-missing values per column
$$\text{Completeness}(col) = \frac{\text{non-null values}}{\text{total rows}}$$

**2. Consistency** — do calories match the macronutrient formula?
$$\text{Expected kcal} = \text{protein} \times 4 + \text{carbs} \times 4 + \text{fat} \times 9$$

**3. Range validity** — are nutritional values within physiologically plausible bounds?


In [ ]:
# ── 4.1 Completeness report ────────────────────────────────────────────────────
check_cols = ['calories','total_fat_g','sugar_g','sodium_mg',
               'protein_g','sat_fat_g','carbs_g','fiber_g']

completeness_rows = []
for col in check_cols:
    present  = df_recipes[col].notna().sum()
    missing  = len(df_recipes) - present
    pct      = present / len(df_recipes)
    status   = 'OK' if pct >= 0.95 else 'Review' if pct >= 0.70 else 'Poor'
    action   = 'Keep' if pct >= 0.95 else 'Mean impute' if pct >= 0.70 else 'Drop or re-scrape'
    completeness_rows.append({
        'Column': col, 'Present': present, 'Missing': missing,
        'Completeness': f'{pct:.1%}', 'Status': status, 'Action': action
    })

df_completeness = pd.DataFrame(completeness_rows)
print('=== COMPLETENESS REPORT ===')
print(df_completeness.to_string(index=False))


In [ ]:
# ── 4.2 Consistency check ─────────────────────────────────────────────────────
TOLERANCE_KCAL = 100  # kcal — Food.com nutrition is approximate

df_quality = df_recipes[['id','name','calories','protein_g','carbs_g',
                           'total_fat_g','sat_fat_g']].copy()
df_quality = df_quality.dropna(subset=['calories','protein_g','carbs_g','total_fat_g'])

df_quality['expected_kcal'] = (
    df_quality['protein_g']   * 4 +
    df_quality['carbs_g']     * 4 +
    df_quality['total_fat_g'] * 9
).round(0)

df_quality['kcal_error']  = (df_quality['calories'] - df_quality['expected_kcal']).abs()
df_quality['consistent']  = df_quality['kcal_error'] <= TOLERANCE_KCAL

n_inconsistent = (~df_quality['consistent']).sum()
pct_inc        = n_inconsistent / len(df_quality)

print(f'Consistency check (tolerance = ±{TOLERANCE_KCAL} kcal):')
print(f'  Checked:       {len(df_quality):,}')
print(f'  Consistent:    {df_quality["consistent"].sum():,}  ({1-pct_inc:.1%})')
print(f'  Inconsistent:  {n_inconsistent:,}  ({pct_inc:.1%})')
print()
print('Worst inconsistencies (largest calorie errors):')
worst = df_quality.nlargest(5, 'kcal_error')[['name','calories','expected_kcal','kcal_error']]
print(worst.to_string(index=False))


In [ ]:
# ── 4.3 Range validity ────────────────────────────────────────────────────────
VALID_RANGES = {
    'calories':    (10,   3000),
    'protein_g':   (0,    300),
    'carbs_g':     (0,    500),
    'total_fat_g': (0,    300),
    'sodium_mg':   (0,   15000),
    'sugar_g':     (0,    400),
}

print('=== RANGE VALIDITY REPORT ===')
flagged_ids = set()
for col, (lo, hi) in VALID_RANGES.items():
    vals    = df_recipes[col].dropna()
    n_out   = ((vals < lo) | (vals > hi)).sum()
    bad_idx = df_recipes[(df_recipes[col].notna()) &
                          ((df_recipes[col] < lo) | (df_recipes[col] > hi))].index
    flagged_ids.update(bad_idx)
    status = 'OK' if n_out == 0 else f'FLAG: {n_out} rows'
    print(f'  {col:<15}  range [{lo}, {hi}]   {status}')

print(f'\nTotal recipes flagged for at least one range violation: {len(flagged_ids)}')


In [ ]:
# ── 4.4 Clean the dataset ─────────────────────────────────────────────────────
df_clean = df_recipes.copy()

# Remove range-violated rows
df_clean = df_clean.drop(index=list(flagged_ids))
print(f'After removing range violations: {len(df_clean):,} recipes')

# Remove calorie-inconsistent rows
bad_consistency_ids = df_quality[~df_quality['consistent']]['id'].values
df_clean = df_clean[~df_clean['id'].isin(bad_consistency_ids)]
print(f'After removing inconsistent kcal:  {len(df_clean):,} recipes')

# Mean impute remaining missing numeric values
print('\nMean imputation for missing values:')
for col in check_cols:
    n_missing = df_clean[col].isna().sum()
    if n_missing > 0:
        col_mean = df_clean[col].mean()
        df_clean[col] = df_clean[col].fillna(col_mean)
        print(f'  {col:<15}  {n_missing} values → imputed with mean {col_mean:.2f}')

df_clean.to_csv('data/recipes_clean.csv', index=False)
print(f'\nClean dataset saved: {len(df_clean):,} recipes')


In [ ]:
# ── 4.5 Data quality visualisation ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Completeness
comp_vals = [df_recipes[c].notna().mean() for c in check_cols]
colors_c  = ['#0F6E56' if v>=0.95 else '#BA7517' if v>=0.70 else '#A32D2D' for v in comp_vals]
axes[0].barh(check_cols, comp_vals, color=colors_c)
axes[0].axvline(0.95, color='red', linestyle='--', linewidth=1, label='95% threshold')
axes[0].set_xlim(0,1.05)
axes[0].set_title('Column Completeness', fontweight='bold')
axes[0].legend(fontsize=8)
for i,v in enumerate(comp_vals):
    axes[0].text(v+0.01, i, f'{v:.0%}', va='center', fontsize=8)

# Calorie error distribution
axes[1].hist(df_quality['kcal_error'].clip(upper=500), bins=50,
              color='#534AB7', alpha=0.8, edgecolor='white')
axes[1].axvline(TOLERANCE_KCAL, color='red', linestyle='--',
                linewidth=1.5, label=f'Tolerance {TOLERANCE_KCAL} kcal')
axes[1].set_xlabel('|Actual kcal - Expected kcal|')
axes[1].set_title('Calorie Consistency Errors', fontweight='bold')
axes[1].legend(fontsize=8)

# Rating distribution
rating_dist = df_inter['rating'].value_counts().sort_index()
axes[2].bar(rating_dist.index, rating_dist.values, color='#0F6E56', edgecolor='white')
axes[2].set_xlabel('Rating')
axes[2].set_title('User Rating Distribution', fontweight='bold')
for i,(k,v) in enumerate(rating_dist.items()):
    axes[2].text(k, v+5000, f'{v/len(df_inter):.0%}', ha='center', fontsize=8)

plt.suptitle('Data Quality Report — Food.com Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/01_data_quality.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 5 — Exploratory Data Analysis


In [ ]:
# ── 5.1 Nutrition distributions ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

plot_cols = ['calories','protein_g','carbs_g','total_fat_g','sodium_mg','sugar_g']
colors    = ['#534AB7','#0F6E56','#BA7517','#993C1D','#185FA5','#A32D2D']

for ax, col, color in zip(axes, plot_cols, colors):
    data = df_clean[col].dropna()
    # Clip at 99th percentile to avoid extreme outliers squashing the chart
    data = data.clip(upper=data.quantile(0.99))
    ax.hist(data, bins=60, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1,
               label=f'Median: {data.median():.0f}')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Nutritional Value Distributions (clipped at 99th pct)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_nutrition_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.2 Health label co-occurrence heatmap ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label frequency
label_freq = df_clean[LABEL_COLS].mean().sort_values(ascending=True)
axes[0].barh(label_freq.index, label_freq.values,
              color=['#0F6E56' if v > 0.2 else '#BA7517' for v in label_freq.values])
axes[0].set_xlabel('Fraction of recipes')
axes[0].set_title('Health Label Frequency', fontweight='bold')
for i, v in enumerate(label_freq.values):
    axes[0].text(v+0.003, i, f'{v:.1%}', va='center', fontsize=8)

# Correlation between labels
label_corr = df_clean[LABEL_COLS].corr()
sns.heatmap(label_corr, annot=True, fmt='.2f', cmap='RdYlGn',
             center=0, ax=axes[1], linewidths=0.3,
             annot_kws={'size': 7}, vmin=-1, vmax=1)
axes[1].set_title('Health Label Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/03_label_analysis.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.3 Ratings overview ─────────────────────────────────────────────────────
df_inter_clean = df_inter[df_inter['recipe_id'].isin(df_clean['id'])].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Ratings per recipe
ratings_per_recipe = df_inter_clean.groupby('recipe_id').size()
axes[0].hist(ratings_per_recipe.clip(upper=ratings_per_recipe.quantile(0.99)),
              bins=50, color='#534AB7', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Number of ratings per recipe')
axes[0].set_title('Ratings Per Recipe', fontweight='bold')

# Ratings per user
ratings_per_user = df_inter_clean.groupby('user_id').size()
axes[1].hist(ratings_per_user.clip(upper=50), bins=50,
              color='#0F6E56', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Number of ratings per user')
axes[1].set_title('Ratings Per User (clipped at 50)', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/04_interaction_overview.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Interaction matrix density: '
      f'{len(df_inter_clean) / (df_inter_clean["user_id"].nunique() * len(df_clean)):.4%}')


---
## Section 6 — Vector Embeddings
**Work Package: Vector Embeddings**

Each recipe becomes a vector combining normalised nutritional features and binary health labels.

$$\mathbf{r}_i = \left[\frac{\text{cal}}{2000},\ \frac{\text{prot}}{150},\ \frac{\text{carbs}}{300},\ \frac{\text{fat}}{100},\ \frac{\text{sodium}}{5000},\ \frac{\text{sugar}}{200},\ \text{label}_1,\ \ldots,\ \text{label}_k\right]$$


In [ ]:
# ── 6.1 Build recipe feature matrix R ────────────────────────────────────────
FEATURE_MAX = {
    'calories':    2000,
    'protein_g':   150,
    'carbs_g':     300,
    'total_fat_g': 100,
    'sodium_mg':   5000,
    'sugar_g':     200,
}
NUMERIC_FEATURES = list(FEATURE_MAX.keys())
ALL_FEATURES     = NUMERIC_FEATURES + LABEL_COLS

def build_recipe_matrix(df: pd.DataFrame) -> tuple:
    nutrition = df[NUMERIC_FEATURES].copy()
    for col, mx in FEATURE_MAX.items():
        nutrition[col] = (nutrition[col].fillna(0) / mx).clip(0, 1)
    labels   = df[LABEL_COLS].fillna(0).astype(float)
    feat_df  = pd.concat([nutrition, labels], axis=1)
    return feat_df.values, list(feat_df.columns), list(df['id'])

R, FEATURE_NAMES, RECIPE_IDS = build_recipe_matrix(df_clean)
# Build a fast lookup: recipe_id -> row index
RID_TO_IDX = {rid: i for i, rid in enumerate(RECIPE_IDS)}

print(f'Recipe matrix R shape: {R.shape}')
print(f'Features: {FEATURE_NAMES}')
print()
print('Example — first recipe vector:')
print(f'  Name: {df_clean.iloc[0]["name"]}')
for name, val in zip(FEATURE_NAMES, R[0]):
    print(f'  {name:<18} {val:.4f}')


In [ ]:
# ── 6.2 User health profile vectors ─────────────────────────────────────────
# A user vector has the same dimensions as a recipe vector.
# Nutritional targets are normalised with the same FEATURE_MAX.
# Health condition flags directly set the label dimensions.

def user_vector(target_cal, target_prot, max_carbs, max_fat,
                max_sodium, max_sugar,
                diabetic=False, low_sodium=False, low_cal=False,
                high_prot=False, low_fat=False, high_fiber=False,
                heart_healthy=False, vegetarian=False,
                vegan=False, gluten_free=False, dairy_free=False) -> np.ndarray:
    nutrition = np.array([
        target_cal  / FEATURE_MAX['calories'],
        target_prot / FEATURE_MAX['protein_g'],
        max_carbs   / FEATURE_MAX['carbs_g'],
        max_fat     / FEATURE_MAX['total_fat_g'],
        max_sodium  / FEATURE_MAX['sodium_mg'],
        max_sugar   / FEATURE_MAX['sugar_g'],
    ])
    labels = np.array([
        float(diabetic), float(low_sodium), float(low_cal),
        float(high_prot), float(low_fat), float(high_fiber),
        float(heart_healthy), float(vegetarian),
        float(vegan), float(gluten_free), float(dairy_free),
    ])
    return np.concatenate([np.clip(nutrition, 0, 1), labels])


DEMO_USERS = {
    'alice':  {
        'profile': 'Type 2 diabetic, low-carb goal, high protein',
        'vec': user_vector(400, 40, 45,  30, 600, 10,
                            diabetic=True, high_prot=True)
    },
    'bob':    {
        'profile': 'Healthy adult, no restrictions, high calorie',
        'vec': user_vector(700, 35, 200, 60, 2000, 80)
    },
    'carol':  {
        'profile': 'Vegan, heart-healthy, low sodium',
        'vec': user_vector(400, 20, 200, 20, 400, 40,
                            vegetarian=True, vegan=True,
                            heart_healthy=True, low_sodium=True)
    },
    'david':  {
        'profile': 'Hypertensive, low sodium required',
        'vec': user_vector(500, 30, 150, 40, 300, 40,
                            low_sodium=True, heart_healthy=True)
    },
    'emma':   {
        'profile': 'Weight loss, low calorie low fat',
        'vec': user_vector(300, 25, 100, 10, 800, 20,
                            low_cal=True, low_fat=True)
    },
}

print('User profiles:')
for name, info in DEMO_USERS.items():
    print(f'  {name:<8} {info["profile"]}')


In [ ]:
# ── 6.3 PCA visualisation of embedding space ─────────────────────────────────
pca  = PCA(n_components=2, random_state=42)
R_2d = pca.fit_transform(R)

fig, ax = plt.subplots(figsize=(11, 7))

# Colour by diabetic_ok label
diab_mask = df_clean['diabetic_ok'].values == 1
ax.scatter(R_2d[~diab_mask, 0], R_2d[~diab_mask, 1],
            c='#D3D1C7', alpha=0.3, s=8, label='Not diabetic-ok')
ax.scatter(R_2d[diab_mask,  0], R_2d[diab_mask,  1],
            c='#0F6E56', alpha=0.6, s=10, label='Diabetic-ok')

# Plot user vectors
U_2d = pca.transform(np.array([u['vec'] for u in DEMO_USERS.values()]))
for i, (uname, info) in enumerate(DEMO_USERS.items()):
    ax.scatter(U_2d[i,0], U_2d[i,1], c='#A32D2D', s=180, marker='*', zorder=5)
    ax.annotate(uname, (U_2d[i,0], U_2d[i,1]),
                xytext=(6,6), textcoords='offset points',
                fontsize=9, fontweight='bold', color='#A32D2D')

ax.set_title('Recipe Embedding Space (PCA 2D)\nGreen = diabetic-ok recipes  |  Stars = user vectors',
              fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plots/05_embedding_space.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 7 — Content-Based Filtering
**Work Package: Recommender System**

Recommend recipes by computing cosine similarity between the user vector and every recipe vector:
$$\text{score}(u, r) = \cos(\mathbf{u}, \mathbf{r}) = \frac{\mathbf{u} \cdot \mathbf{r}}{\|\mathbf{u}\| \cdot \|\mathbf{r}\|}$$


In [ ]:
# ── 7.1 Content-based scoring ────────────────────────────────────────────────
def cb_scores(user_vec: np.ndarray) -> np.ndarray:
    """Return cosine similarity between user_vec and every recipe in R."""
    return cosine_similarity(user_vec.reshape(1,-1), R).flatten()


def cb_recommend(user_name: str, k: int = 10) -> pd.DataFrame:
    scores = cb_scores(DEMO_USERS[user_name]['vec'])
    result = df_clean[['id','name','calories','protein_g','carbs_g',
                         'total_fat_g','sodium_mg'] + LABEL_COLS].copy()
    result['cb_score'] = scores
    return result.nlargest(k, 'cb_score').reset_index(drop=True)


print('=== Content-Based Top 10 for Alice (diabetic, high-protein) ===')
alice_cb = cb_recommend('alice', k=10)
display_cols = ['name','calories','protein_g','carbs_g','sodium_mg','diabetic_ok','cb_score']
print(alice_cb[display_cols].to_string(index=False))


In [ ]:
# ── 7.2 Score matrix heatmap ─────────────────────────────────────────────────
S_content = np.array([cb_scores(info['vec']) for info in DEMO_USERS.values()])

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(S_content, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_yticks(range(len(DEMO_USERS)))
ax.set_yticklabels([f'{n}  ({DEMO_USERS[n]["profile"][:35]})' for n in DEMO_USERS])
ax.set_xlabel('Recipe index (sorted by ID)')
ax.set_title('Content-Based Score Matrix  S[user, recipe]', fontweight='bold')
plt.colorbar(im, ax=ax, label='Cosine similarity')
plt.tight_layout()
plt.savefig('plots/06_score_matrix_content.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 8 — Collaborative Filtering (SVD Matrix Factorisation)
**Work Package: Recommender System**

Matrix factorisation decomposes the sparse rating matrix:
$$R \approx U \times V^T$$
- $U$ = (n\_users × k) user latent factors
- $V$ = (n\_recipes × k) recipe latent factors
- $k$ = number of hidden taste dimensions

Training minimises:
$$\mathcal{L} = \sum_{(u,r)\text{ known}} (R_{ur} - \mathbf{u}_u \cdot \mathbf{v}_r)^2 + \lambda(\|U\|^2 + \|V\|^2)$$


In [ ]:
# ── 8.1 Prepare Surprise dataset ─────────────────────────────────────────────
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as svd_split

# Filter interactions to only clean recipes
clean_ids = set(df_clean['id'].values)
df_inter_clean = df_inter[df_inter['recipe_id'].isin(clean_ids)].copy()

# Surprise needs integer user/item IDs — map them
unique_users   = df_inter_clean['user_id'].unique()
unique_recipes = df_inter_clean['recipe_id'].unique()
user2idx       = {u: i for i, u in enumerate(unique_users)}
recipe2idx     = {r: i for i, r in enumerate(unique_recipes)}
idx2recipe     = {i: r for r, i in recipe2idx.items()}

df_inter_mapped = df_inter_clean.copy()
df_inter_mapped['user_idx']   = df_inter_mapped['user_id'].map(user2idx)
df_inter_mapped['recipe_idx'] = df_inter_mapped['recipe_id'].map(recipe2idx)

reader   = Reader(rating_scale=(1, 5))
data_svd = Dataset.load_from_df(
    df_inter_mapped[['user_idx','recipe_idx','rating']], reader
)
trainset, testset = svd_split(data_svd, test_size=0.2, random_state=42)

print(f'Training set: {trainset.n_ratings:,} ratings')
print(f'Test set:     {len(testset):,} ratings')
print(f'Users:        {trainset.n_users:,}')
print(f'Recipes:      {trainset.n_items:,}')


In [ ]:
# ── 8.2 Train SVD ─────────────────────────────────────────────────────────────
print('Training SVD matrix factorisation...')
svd_model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.1, random_state=42)
svd_model.fit(trainset)

predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)

print(f'\nSVD evaluation on test set:')
print(f'  RMSE: {rmse:.4f}')
print(f'  MAE:  {mae:.4f}')
print()
print('Sample predictions vs actual (first 10):')
print(f'{"User":>10} {"Recipe":>10} {"Actual":>8} {"Predicted":>10}')
for pred in predictions[:10]:
    print(f'{pred.uid:>10} {str(pred.iid):>10} {pred.r_ui:>8.1f} {pred.est:>10.3f}')


In [ ]:
# ── 8.3 Collaborative filtering recommendations for a known user ───────────────
def cf_recommend(user_orig_id: int, k: int = 10) -> pd.DataFrame:
    """
    Recommend top-k unseen recipes for a user who has ratings in the dataset.
    user_orig_id: the original Food.com user_id
    """
    if user_orig_id not in user2idx:
        print(f'User {user_orig_id} not in training data')
        return pd.DataFrame()

    uid = user2idx[user_orig_id]
    already_rated = set(
        df_inter_clean[df_inter_clean['user_id']==user_orig_id]['recipe_id']
    )
    unseen_recipes = [rid for rid in unique_recipes if rid not in already_rated]

    preds = [(rid, svd_model.predict(uid, recipe2idx[rid]).est)
              for rid in unseen_recipes if rid in recipe2idx]
    preds = sorted(preds, key=lambda x: x[1], reverse=True)[:k]

    rec_ids = [p[0] for p in preds]
    scores  = {p[0]: p[1] for p in preds}
    result  = df_clean[df_clean['id'].isin(rec_ids)].copy()
    result['cf_score'] = result['id'].map(scores)
    return result.sort_values('cf_score', ascending=False).reset_index(drop=True)


# Pick a real user from the dataset
sample_user = df_inter_clean['user_id'].value_counts().index[5]  # 6th most active user
print(f'Sample user ID: {sample_user}')
print(f'Their rating count: {(df_inter_clean["user_id"]==sample_user).sum()}')
print()
cf_recs = cf_recommend(sample_user, k=10)
if len(cf_recs):
    print('Top 10 CF recommendations:')
    print(cf_recs[['name','calories','protein_g','carbs_g','cf_score']].to_string(index=False))


In [ ]:
# ── 8.4 Save model ────────────────────────────────────────────────────────────
import pickle
with open('models/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_model, f)
np.save('models/recipe_matrix.npy', R)
print('Models saved to models/')


---
## Section 9 — Hybrid Recommender + Health Constraint Filter
**Work Package: Recommender System**

Blend both signals:
$$\text{final\_score}(u,r) = \alpha \cdot \text{cb\_score}(u,r) + (1-\alpha) \cdot \text{cf\_score}(u,r)$$

Then apply hard health rules that **cannot be overridden by the model**.


In [ ]:
# ── 9.1 Hybrid scoring ────────────────────────────────────────────────────────
def hybrid_scores(user_name: str,
                   user_orig_id: int = None,
                   alpha: float = 0.6) -> np.ndarray:
    """
    Blend content-based (alpha) and collaborative (1-alpha) scores.
    If user has no CF history, falls back to pure content-based.
    """
    cb = cb_scores(DEMO_USERS[user_name]['vec'])
    cb_norm = (cb - cb.min()) / (cb.max() - cb.min() + 1e-9)

    if user_orig_id is not None and user_orig_id in user2idx:
        uid = user2idx[user_orig_id]
        cf_raw = np.array([
            svd_model.predict(uid, recipe2idx.get(rid, -1)).est
            if rid in recipe2idx else 3.0
            for rid in RECIPE_IDS
        ])
        cf_norm = (cf_raw - cf_raw.min()) / (cf_raw.max() - cf_raw.min() + 1e-9)
    else:
        cf_norm = cb_norm   # cold start fallback
        alpha   = 1.0

    return alpha * cb_norm + (1 - alpha) * cf_norm


# ── 9.2 Health constraint filter ─────────────────────────────────────────────
HEALTH_CONSTRAINTS = {
    'alice':  {'diabetic': True,  'hypertensive': False, 'vegan': False, 'gluten_free': False},
    'bob':    {'diabetic': False, 'hypertensive': False, 'vegan': False, 'gluten_free': False},
    'carol':  {'diabetic': False, 'hypertensive': False, 'vegan': True,  'gluten_free': False},
    'david':  {'diabetic': False, 'hypertensive': True,  'vegan': False, 'gluten_free': False},
    'emma':   {'diabetic': False, 'hypertensive': False, 'vegan': False, 'gluten_free': False},
}

def health_filter(df_recs: pd.DataFrame, constraints: dict) -> pd.DataFrame:
    df = df_recs.copy()
    df['blocked_reason'] = ''
    if constraints.get('diabetic'):
        m = (df['carbs_g'] > 45) | (df['sugar_g'] > 10)
        df.loc[m, 'blocked_reason'] += 'high-carb/sugar; '
    if constraints.get('hypertensive'):
        m = df['sodium_mg'] > 600
        df.loc[m, 'blocked_reason'] += 'high-sodium; '
    if constraints.get('vegan'):
        m = df['vegan'] == 0
        df.loc[m, 'blocked_reason'] += 'not-vegan; '
    if constraints.get('gluten_free'):
        m = df['gluten_free'] == 0
        df.loc[m, 'blocked_reason'] += 'contains-gluten; '
    df['allowed'] = df['blocked_reason'] == ''
    return df


def full_recommend(user_name: str, user_orig_id: int = None,
                    alpha: float = 0.6, k: int = 10) -> pd.DataFrame:
    """Full pipeline: hybrid score → health filter → top-k."""
    scores  = hybrid_scores(user_name, user_orig_id, alpha)
    result  = df_clean[['id','name','calories','protein_g','carbs_g',
                          'total_fat_g','sodium_mg','sugar_g'] + LABEL_COLS].copy()
    result['hybrid_score'] = scores
    # Take top-200 candidates before filtering
    candidates = result.nlargest(200, 'hybrid_score')
    filtered   = health_filter(candidates, HEALTH_CONSTRAINTS[user_name])
    return filtered[filtered['allowed']].head(k).reset_index(drop=True)


# Show results for all demo users
for uname in DEMO_USERS:
    recs = full_recommend(uname, k=5)
    print(f'\n{uname.upper()} — {DEMO_USERS[uname]["profile"]}')
    print(recs[['name','calories','protein_g','carbs_g','sodium_mg','hybrid_score']]
          .to_string(index=False))


In [ ]:
# ── 9.3 Visualise health filter effect for Alice ──────────────────────────────
scores_alice  = hybrid_scores('alice', alpha=0.6)
result_alice  = df_clean[['id','name','calories','protein_g','carbs_g',
                            'sodium_mg','sugar_g'] + LABEL_COLS].copy()
result_alice['hybrid_score'] = scores_alice
top20_alice   = result_alice.nlargest(20, 'hybrid_score')
filtered_alice = health_filter(top20_alice, HEALTH_CONSTRAINTS['alice'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_before = ['#A32D2D' if not a else '#0F6E56'
                  for a in filtered_alice['allowed']]
axes[0].barh(filtered_alice['name'], filtered_alice['hybrid_score'],
              color=colors_before)
axes[0].set_title('BEFORE health filter\n(red = blocked for Alice: diabetic)',
                   fontweight='bold')
axes[0].set_xlabel('Hybrid score')
axes[0].invert_yaxis()

allowed_alice = filtered_alice[filtered_alice['allowed']].head(10)
axes[1].barh(allowed_alice['name'], allowed_alice['hybrid_score'],
              color='#0F6E56')
axes[1].set_title('AFTER health filter\n(carbs ≤ 45g AND sugar ≤ 10g)',
                   fontweight='bold')
axes[1].set_xlabel('Hybrid score')
axes[1].invert_yaxis()

plt.suptitle('Health Constraint Filtering — Alice (Type 2 Diabetic)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/07_health_filter.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 10 — Perturbation Analysis
**Work Package: Perturbation Analysis**

We test robustness by introducing controlled noise to the user input and measuring
how much the top-k recommendation list changes.

**Metric — Jaccard similarity:**
$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
- $J = 1.0$ → identical lists (perfectly robust)
- $J = 0.0$ → completely different lists (fragile)


In [ ]:
# ── 10.1 Perturbation functions ──────────────────────────────────────────────
def jaccard(a: set, b: set) -> float:
    if not a and not b: return 1.0
    return len(a & b) / len(a | b)


def perturb_numeric(vec: np.ndarray, sigma: float = 0.05) -> np.ndarray:
    """Add Gaussian noise to the nutritional (numeric) part of the user vector."""
    p = vec.copy()
    p[:6] += np.random.normal(0, sigma, 6)
    return np.clip(p, 0, 1)


def perturb_flip_flag(vec: np.ndarray, flag_position: int) -> np.ndarray:
    """Flip one binary health flag in the user vector."""
    p = vec.copy()
    p[6 + flag_position] = 1 - p[6 + flag_position]
    return p


def get_topk_ids(user_vec: np.ndarray, k: int = 10) -> set:
    scores = cosine_similarity(user_vec.reshape(1,-1), R).flatten()
    return set(np.array(RECIPE_IDS)[np.argsort(scores)[::-1][:k]])


# ── 10.2 Run perturbation experiments for all users ───────────────────────────
N_TRIALS = 50
SIGMA_LEVELS = [0.01, 0.03, 0.05, 0.10, 0.20]
FLAG_NAMES   = ['diabetic','low_sodium','low_calorie','high_protein',
                 'low_fat','high_fiber','heart_healthy','vegetarian']

perturb_results = []

for uname, info in DEMO_USERS.items():
    base_vec = info['vec']
    base_top = get_topk_ids(base_vec, k=10)

    for sigma in SIGMA_LEVELS:
        jaccards = [jaccard(base_top, get_topk_ids(perturb_numeric(base_vec, sigma)))
                    for _ in range(N_TRIALS)]
        perturb_results.append({
            'user': uname, 'perturbation': f'numeric noise σ={sigma}',
            'mean_J': np.mean(jaccards), 'std_J': np.std(jaccards)
        })

    for fi, fname in enumerate(FLAG_NAMES[:4]):
        p_vec = perturb_flip_flag(base_vec, fi)
        j     = jaccard(base_top, get_topk_ids(p_vec, k=10))
        perturb_results.append({
            'user': uname, 'perturbation': f'flip: {fname}',
            'mean_J': j, 'std_J': 0.0
        })

df_perturb = pd.DataFrame(perturb_results)
print('=== Perturbation Analysis Summary ===')
pivot = df_perturb.pivot_table(index='perturbation', columns='user',
                                 values='mean_J', aggfunc='mean')
print(pivot.round(3).to_string())


In [ ]:
# ── 10.3 Visualise ────────────────────────────────────────────────────────────
alice_perturb = df_perturb[df_perturb['user']=='alice']

fig, ax = plt.subplots(figsize=(10, 5))
colors_p = ['#0F6E56' if v>=0.7 else '#BA7517' if v>=0.4 else '#A32D2D'
              for v in alice_perturb['mean_J']]
bars = ax.bar(alice_perturb['perturbation'], alice_perturb['mean_J'],
               color=colors_p, edgecolor='white',
               yerr=alice_perturb['std_J'],
               capsize=3, error_kw={'color':'#888780','linewidth':1})
ax.axhline(0.7, color='#0F6E56', linestyle='--', linewidth=1.2, label='Robust (J ≥ 0.7)')
ax.axhline(0.4, color='#A32D2D', linestyle='--', linewidth=1.2, label='Fragile (J < 0.4)')
ax.set_ylabel('Jaccard similarity (higher = more stable)')
ax.set_title('Perturbation Robustness — Alice', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=8)
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('plots/08_perturbation.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 11 — Performance Evaluation
**Work Package: Performance Evaluation**

**Two evaluation methods** (required by the work package):

- **Method 1: Holdout split** — train on 80% of ratings, evaluate on 20%
- **Method 2: Leave-one-out** — hide each user's best-rated recipe, check if model finds it in top-k

$$\text{Precision}@k = \frac{|\text{relevant} \cap \text{recommended}|}{k}$$

$$\text{Recall}@k = \frac{|\text{relevant} \cap \text{recommended}|}{|\text{relevant}|}$$


In [ ]:
# ── 11.1 Build relevance sets (ratings >= 4 = liked) ─────────────────────────
RELEVANCE_THRESHOLD = 4

# Use a random sample of 500 users for faster evaluation
eval_users = df_inter_clean['user_id'].value_counts()
eval_users = eval_users[eval_users >= 5].index[:500]  # users with >=5 ratings

def get_relevant(user_id: int) -> set:
    return set(df_inter_clean[
        (df_inter_clean['user_id'] == user_id) &
        (df_inter_clean['rating']  >= RELEVANCE_THRESHOLD)
    ]['recipe_id'])

relevant_sets = {uid: get_relevant(uid) for uid in eval_users}
# Only keep users who actually have relevant items
relevant_sets = {uid: rset for uid, rset in relevant_sets.items() if len(rset) > 0}
print(f'Evaluation users with ≥1 relevant recipe: {len(relevant_sets)}')


In [ ]:
# ── 11.2 Precision@k, Recall@k, F1@k ────────────────────────────────────────
def precision_at_k(rec_ids: list, relevant: set, k: int) -> float:
    hits = len(set(rec_ids[:k]) & relevant)
    return hits / k if k else 0.0

def recall_at_k(rec_ids: list, relevant: set, k: int) -> float:
    hits = len(set(rec_ids[:k]) & relevant)
    return hits / len(relevant) if relevant else 0.0

def f1_at_k(p: float, r: float) -> float:
    return 2*p*r/(p+r) if (p+r) else 0.0


def evaluate_cf(k_values=(1,3,5,10,15,20), max_users=500) -> pd.DataFrame:
    """Evaluate the SVD collaborative filtering model."""
    rows = []
    user_sample = list(relevant_sets.keys())[:max_users]

    for k in k_values:
        p_list, r_list, f_list = [], [], []
        for uid in user_sample:
            if uid not in user2idx: continue
            ui = user2idx[uid]
            already_rated = set(df_inter_clean[df_inter_clean['user_id']==uid]['recipe_id'])
            unseen = [rid for rid in unique_recipes
                       if rid not in already_rated and rid in recipe2idx]
            preds  = sorted(
                [(rid, svd_model.predict(ui, recipe2idx[rid]).est) for rid in unseen],
                key=lambda x: x[1], reverse=True
            )
            rec_ids = [p[0] for p in preds[:max(k_values)]]
            rel = relevant_sets[uid]
            p = precision_at_k(rec_ids, rel, k)
            r = recall_at_k(rec_ids, rel, k)
            p_list.append(p); r_list.append(r); f_list.append(f1_at_k(p,r))

        rows.append({'k':k,'method':'CF (SVD)',
                      'precision':np.mean(p_list),
                      'recall':   np.mean(r_list),
                      'f1':       np.mean(f_list)})
    return pd.DataFrame(rows)


def evaluate_cb(k_values=(1,3,5,10,15,20)) -> pd.DataFrame:
    """Evaluate content-based filtering (using demo user alice as proxy)."""
    rows = []
    # For CB we can only evaluate users whose health profile we know
    # Here we evaluate against all users' actual ratings using the alice vector
    # In a real system users would provide their health profiles
    user_sample = list(relevant_sets.keys())[:200]

    for k in k_values:
        p_list, r_list, f_list = [], [], []
        for uid in user_sample:
            # Use the actual ratings to build a user vector proxy
            user_ratings = df_inter_clean[
                df_inter_clean['user_id'] == uid
            ].sort_values('rating', ascending=False)
            if len(user_ratings) == 0: continue

            # Build user vector as mean of top-rated recipe vectors
            top_ids = user_ratings[user_ratings['rating']>=4]['recipe_id'].values[:5]
            valid_idx = [RID_TO_IDX[rid] for rid in top_ids if rid in RID_TO_IDX]
            if not valid_idx: continue
            uv = R[valid_idx].mean(axis=0)

            scores  = cosine_similarity(uv.reshape(1,-1), R).flatten()
            order   = np.argsort(scores)[::-1]
            rec_ids = [RECIPE_IDS[i] for i in order]
            rel     = relevant_sets[uid]
            p = precision_at_k(rec_ids, rel, k)
            r = recall_at_k(rec_ids, rel, k)
            p_list.append(p); r_list.append(r); f_list.append(f1_at_k(p,r))

        rows.append({'k':k,'method':'CB (cosine)',
                      'precision':np.mean(p_list),
                      'recall':   np.mean(r_list),
                      'f1':       np.mean(f_list)})
    return pd.DataFrame(rows)


print('Evaluating CF model (this takes ~1-2 min for 500 users)...')
eval_cf = evaluate_cf()
print('Evaluating CB model...')
eval_cb = evaluate_cb()
eval_all = pd.concat([eval_cf, eval_cb], ignore_index=True)

print('\n=== EVALUATION RESULTS ===')
print(eval_all.pivot_table(index='k', columns='method',
      values=['precision','recall','f1']).round(4).to_string())


In [ ]:
# ── 11.3 Plot Precision-Recall curves ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
method_colors = {'CF (SVD)': '#534AB7', 'CB (cosine)': '#0F6E56'}

for ax, metric in zip(axes, ['precision','recall','f1']):
    for method, color in method_colors.items():
        sub = eval_all[eval_all['method']==method]
        ax.plot(sub['k'], sub[metric], marker='o', color=color,
                 linewidth=1.8, markersize=5, label=method)
    ax.set_xlabel('k')
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f'{metric.capitalize()}@k', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Recommender Evaluation — Food.com Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/09_evaluation_curves.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 11.4 Method 2: Leave-one-out evaluation ───────────────────────────────────
print('=== Method 2: Leave-One-Out Evaluation ===')
loo_hits = []
loo_sample = list(relevant_sets.keys())[:200]

for uid in loo_sample:
    if uid not in user2idx: continue
    ui = user2idx[uid]

    user_ratings = df_inter_clean[
        df_inter_clean['user_id'] == uid
    ].sort_values('rating', ascending=False)
    if len(user_ratings) < 2: continue

    # Hold out the highest-rated recipe
    held_out_id = user_ratings.iloc[0]['recipe_id']
    if held_out_id not in recipe2idx: continue

    unseen = [rid for rid in unique_recipes
               if rid != held_out_id and rid in recipe2idx]
    preds  = sorted(
        [(rid, svd_model.predict(ui, recipe2idx[rid]).est) for rid in unseen],
        key=lambda x: x[1], reverse=True
    )
    top10 = {p[0] for p in preds[:10]}
    loo_hits.append(int(held_out_id in top10))

hit_rate = np.mean(loo_hits)
print(f'Leave-one-out Hit@10: {hit_rate:.1%}  ({sum(loo_hits)}/{len(loo_hits)} users)')
print(f'(How often the held-out top-rated recipe appears in the top-10 recommendations)')


---
## Section 12 — Hyperparameter Tuning with Optuna
**Work Package: Hyperparameter Tuning**

We tune four hyperparameters:

| Parameter | Role | Search range |
|-----------|------|--------------|
| `n_factors` (k) | Latent dimensions of U, V | 10 – 150 |
| `reg_all` (λ) | L2 regularisation | 0.001 – 1.0 |
| `lr_all` | Gradient descent step size | 0.001 – 0.05 |
| `alpha` | Content vs collaborative blend | 0.0 – 1.0 |

Optuna uses **Tree-structured Parzen Estimators (TPE)** — learns from previous trials to focus on promising regions.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Use a small user sample for fast tuning
TUNE_USERS = list(relevant_sets.keys())[:100]

def optuna_objective(trial):
    n_factors = trial.suggest_int('n_factors', 10, 150)
    reg_all   = trial.suggest_float('reg_all',  0.001, 1.0,  log=True)
    lr_all    = trial.suggest_float('lr_all',   0.001, 0.05, log=True)
    alpha     = trial.suggest_float('alpha',    0.0,   1.0)

    # Train SVD with these params
    model = SVD(n_factors=n_factors, reg_all=reg_all, lr_all=lr_all,
                 n_epochs=10, random_state=42)
    model.fit(trainset)

    # Evaluate: mean Precision@10 on tune users
    p_list = []
    for uid in TUNE_USERS:
        if uid not in user2idx: continue
        ui = user2idx[uid]
        rel = relevant_sets.get(uid, set())
        if not rel: continue

        # CF scores
        unseen = [rid for rid in unique_recipes
                   if rid not in set(df_inter_clean[
                       df_inter_clean['user_id']==uid]['recipe_id'])
                   and rid in recipe2idx]
        if not unseen: continue
        cf_raw = np.array([model.predict(ui, recipe2idx[rid]).est for rid in unseen])
        cf_n   = (cf_raw - cf_raw.min()) / (cf_raw.max() - cf_raw.min() + 1e-9)

        # Build CB score from user's top-rated recipes as proxy vector
        top_ids = df_inter_clean[
            (df_inter_clean['user_id']==uid) & (df_inter_clean['rating']>=4)
        ]['recipe_id'].values[:5]
        valid_idx = [RID_TO_IDX[rid] for rid in top_ids if rid in RID_TO_IDX]
        if not valid_idx: continue
        uv    = R[valid_idx].mean(axis=0)
        unseen_idx = [RID_TO_IDX[rid] for rid in unseen if rid in RID_TO_IDX]
        if not unseen_idx: continue
        cb_raw = cosine_similarity(uv.reshape(1,-1), R[unseen_idx]).flatten()
        cb_n   = (cb_raw - cb_raw.min()) / (cb_raw.max() - cb_raw.min() + 1e-9)

        # Use only recipes that appear in both
        n = min(len(cb_n), len(cf_n))
        final = alpha * cb_n[:n] + (1 - alpha) * cf_n[:n]
        rec_ids = [unseen[i] for i in np.argsort(final)[::-1][:10]]
        p_list.append(precision_at_k(rec_ids, rel, 10))

    return np.mean(p_list) if p_list else 0.0


print('Running Optuna (50 trials)...')
study = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(optuna_objective, n_trials=50, show_progress_bar=False)

print(f'\nBest Precision@10: {study.best_value:.4f}')
print('Best hyperparameters:')
for k, v in study.best_params.items():
    print(f'  {k:<15} = {v}')


In [ ]:
# ── 12.2 Retrain with best params ────────────────────────────────────────────
best = study.best_params
best_svd = SVD(
    n_factors = best['n_factors'],
    reg_all   = best['reg_all'],
    lr_all    = best['lr_all'],
    n_epochs  = 30,
    random_state = 42
)
best_svd.fit(trainset)
best_preds = best_svd.test(testset)
print(f'Tuned model RMSE: {accuracy.rmse(best_preds, verbose=False):.4f}')
print(f'Tuned model MAE:  {accuracy.mae(best_preds,  verbose=False):.4f}')

with open('models/svd_best.pkl', 'wb') as f:
    pickle.dump({'model': best_svd, 'params': best}, f)
print('Tuned model saved to models/svd_best.pkl')


In [ ]:
# ── 12.3 Visualise hyperparameter importance ──────────────────────────────────
trials_df = study.trials_dataframe()
params    = ['params_n_factors','params_reg_all','params_lr_all','params_alpha']
labels    = ['n_factors (k)','λ (reg_all)','lr','α (blend)']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for ax, param, label in zip(axes, params, labels):
    if param not in trials_df.columns: continue
    sc = ax.scatter(trials_df[param], trials_df['value'],
                     c=trials_df['value'], cmap='RdYlGn',
                     alpha=0.7, s=40)
    best_val = best.get(param.replace('params_',''), None)
    if best_val:
        ax.axvline(best_val, color='black', linestyle='--',
                    linewidth=1.5, label=f'best={best_val:.4f}')
        ax.legend(fontsize=8)
    ax.set_xlabel(label)
    ax.set_ylabel('Precision@10')
    ax.set_title(f'Effect of {label}', fontweight='bold')
    plt.colorbar(sc, ax=ax)

plt.suptitle('Optuna Hyperparameter Search — Food.com', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/10_hyperparameter_tuning.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 13 — Experiment Logging with Weights & Biases
**Work Package: Experiments Logging**

Every Optuna trial is logged to W&B — hyperparameters, Precision@10, RMSE.

**Setup:** run `wandb login` in your terminal, paste your API key from wandb.ai.


In [ ]:
WANDB_ENABLED = False  # Set True after: pip install wandb && wandb login

if WANDB_ENABLED:
    import wandb
    for trial in study.trials:
        wandb.init(
            project = 'food-recommender-foodcom',
            name    = f'trial-{trial.number:03d}',
            config  = trial.params,
            reinit  = True,
            tags    = ['optuna','svd','hybrid','food.com'],
        )
        wandb.log({'precision_at_10': trial.value, **trial.params})
        wandb.finish()
    # Log best model
    wandb.init(project='food-recommender-foodcom', name='best-model',
                config=best, tags=['final'])
    wandb.log({'precision_at_10': study.best_value,
                'rmse': accuracy.rmse(best_preds, verbose=False),
                'mae':  accuracy.mae( best_preds, verbose=False)})
    wandb.finish()
    print('All trials logged to W&B')
else:
    print('W&B logging off. Set WANDB_ENABLED=True to log.')
    print()
    print(f'Would log {len(study.trials)} trials to project: food-recommender-foodcom')
    print(f'Best trial: precision@10={study.best_value:.4f}  params={study.best_params}')


---
## Section 14 — Streamlit Frontend
**Work Package: Frontend Application**

Run with: `streamlit run app.py`


In [ ]:
APP_CODE = '''
import streamlit as st
import pandas as pd
import numpy as np
import pickle, ast
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px

st.set_page_config(page_title='Food Recommender', page_icon='🥗', layout='wide')

FEATURE_MAX = dict(calories=2000,protein_g=150,carbs_g=300,
                    total_fat_g=100,sodium_mg=5000,sugar_g=200)
LABEL_COLS  = ['diabetic_ok','low_sodium','low_calorie','high_protein',
                'low_fat','high_fiber','heart_healthy','vegetarian',
                'vegan','gluten_free','dairy_free']
NUMERIC     = list(FEATURE_MAX.keys())

@st.cache_data
def load_data():
    df = pd.read_csv("data/recipes_clean.csv")
    return df

@st.cache_resource
def load_model():
    try:
        with open("models/svd_best.pkl","rb") as f:
            return pickle.load(f)
    except:
        return None

def build_R(df):
    nut = df[NUMERIC].copy()
    for c, mx in FEATURE_MAX.items():
        nut[c] = nut[c].fillna(0) / mx
    lbl = df[LABEL_COLS].fillna(0).astype(float)
    return pd.concat([nut, lbl], axis=1).values

def make_user_vec(cal,prot,carbs,fat,sodium,sugar,
                   diabetic,hypertensive,vegan,gf):
    n = np.array([cal/2000,prot/150,carbs/300,fat/100,sodium/5000,sugar/200])
    l = np.array([float(diabetic),float(hypertensive),0,
                   float(prot>=25),float(fat<=10),0,
                   float(hypertensive),float(vegan),
                   float(vegan),float(gf),0])
    return np.clip(np.concatenate([n,l]),0,1)

def health_filter(df_r, diabetic, hypertensive, vegan, gf):
    mask = pd.Series([True]*len(df_r), index=df_r.index)
    if diabetic:     mask &= (df_r["carbs_g"]<=45)&(df_r["sugar_g"]<=10)
    if hypertensive: mask &= (df_r["sodium_mg"]<=600)
    if vegan:        mask &= (df_r["vegan"]==1)
    if gf:           mask &= (df_r["gluten_free"]==1)
    return df_r[mask]

# ── UI ────────────────────────────────────────────────────────
df = load_data()
R  = build_R(df)

st.title("🥗 Health-Personalized Food Recommender")
st.caption("Based on Food.com — 500k+ recipes | personalized to your health profile")

col1, col2 = st.columns([1, 2])

with col1:
    st.subheader("Your Health Profile")
    target_cal   = st.slider("Target calories",       200, 900, 500, 50)
    target_prot  = st.slider("Target protein (g)",      5,  80,  30,  5)
    max_carbs    = st.slider("Max carbs (g)",           10, 300, 150, 10)
    max_fat      = st.slider("Max fat (g)",              5, 100,  40,  5)
    max_sodium   = st.slider("Max sodium (mg)",        100,3000, 800,100)
    max_sugar    = st.slider("Max sugar (g)",            0,  80,  20,  5)
    st.divider()
    diabetic     = st.checkbox("Type 2 Diabetes  (carbs ≤ 45g, sugar ≤ 10g)")
    hypertensive = st.checkbox("Hypertension  (sodium ≤ 600mg)")
    vegan        = st.checkbox("Vegan")
    gluten_free  = st.checkbox("Gluten-free")
    k_recs       = st.slider("Recommendations to show", 3, 20, 8)

with col2:
    user_vec  = make_user_vec(target_cal,target_prot,max_carbs,max_fat,
                               max_sodium,max_sugar,
                               diabetic,hypertensive,vegan,gluten_free)
    scores    = cosine_similarity(user_vec.reshape(1,-1), R).flatten()
    result    = df.copy()
    result["score"] = scores
    result    = result.sort_values("score", ascending=False)
    result    = health_filter(result, diabetic, hypertensive, vegan, gluten_free)
    recs      = result.head(k_recs).reset_index(drop=True)

    st.subheader(f"Top {k_recs} Recommendations")
    if len(recs) == 0:
        st.warning("No recipes match your constraints. Try relaxing some conditions.")
    else:
        fig = px.bar(recs, x="score", y="name", orientation="h",
                      color="score", color_continuous_scale="Teal",
                      hover_data=["calories","protein_g","carbs_g","sodium_mg"],
                      labels={"score":"Match score","name":"Recipe"})
        fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=420,
                           showlegend=False, coloraxis_showscale=False)
        st.plotly_chart(fig, use_container_width=True)

        st.dataframe(
            recs[["name","calories","protein_g","carbs_g",
                   "total_fat_g","sodium_mg","sugar_g"]].round(1),
            use_container_width=True, hide_index=True
        )

with st.sidebar:
    st.subheader("Dataset Info")
    st.metric("Total recipes", f"{len(df):,}")
    st.metric("Diabetic-ok",   f"{int(df["diabetic_ok"].sum()):,}")
    st.metric("Vegan",         f"{int(df["vegan"].sum()):,}")
    st.metric("Gluten-free",   f"{int(df["gluten_free"].sum()):,}")
    st.divider()
    if st.button("Show calorie distribution"):
        import matplotlib.pyplot as plt
        fig2, ax = plt.subplots()
        df["calories"].clip(upper=1500).hist(bins=60, ax=ax, color="#534AB7")
        ax.set_xlabel("Calories")
        st.pyplot(fig2)
'''

with open('app.py', 'w') as f:
    f.write(APP_CODE)
print('app.py written.')
print('Run with:  streamlit run app.py')


---
## Section 15 — Full Pipeline Summary


In [ ]:
import glob
print('=' * 62)
print('HEALTH FOOD RECOMMENDER — PIPELINE SUMMARY')
print('=' * 62)
print(f'Dataset:          Food.com Recipes & User Interactions')
print(f'Recipes (clean):  {len(df_clean):,}')
print(f'Interactions:     {len(df_inter_clean):,}')
print(f'Unique users:     {df_inter_clean["user_id"].nunique():,}')
print()
print(f'SCRAPING        USDA API enriched {len(df_usda)} recipes with fiber/potassium')
print(f'ANNOTATION      {len(LABEL_COLS)} health labels applied to {len(df_clean):,} recipes')
print(f'DATA QUALITY    {len(flagged_ids)} range violations removed')
print(f'                {n_inconsistent} calorie-inconsistent rows removed')
print(f'EMBEDDINGS      Recipe matrix R: {R.shape}')
print(f'SVD MODEL       k={best["n_factors"]}  λ={best["reg_all"]:.4f}  lr={best["lr_all"]:.4f}')
print(f'                RMSE={accuracy.rmse(best_preds, verbose=False):.4f}  '
      f'MAE={accuracy.mae(best_preds, verbose=False):.4f}')
k10_cf = eval_cf[eval_cf['k']==10].iloc[0]
print(f'EVALUATION      CF  Precision@10={k10_cf["precision"]:.4f}  Recall@10={k10_cf["recall"]:.4f}')
print(f'                LOO Hit@10={hit_rate:.1%}')
print(f'PERTURBATION    Mean Jaccard stability: '
      f'{df_perturb[df_perturb["perturbation"].str.startswith("numeric")]["mean_J"].mean():.3f}')
print(f'TUNING          Optuna: {len(study.trials)} trials  '
      f'Best Precision@10={study.best_value:.4f}')
print()
plots = glob.glob('plots/*.png')
print(f'Saved plots ({len(plots)}):')
for p in sorted(plots): print(f'  {p}')
print()
print('FRONTEND: streamlit run app.py')
print('=' * 62)
